# 14. Scaffold split vs random drug split

**Question.** Is the model ranking (RF > CNN > graph) and the cold-drug performance level an accident of one random
drug split, or does it hold when chemically similar drugs are kept together?

**Design.** All 18,037 pancreatic pairs are pooled. Drugs are grouped by Bemis-Murcko scaffold (RDKit); whole scaffold
groups are held out as test (about 21% of drugs), so analogues of a test drug are not in training. Compared with a
random drug-grouped split of the same size, over 3 seeds each. Training size fixed at 12,929 pairs.

**Models.** RF + AE (300 trees) and cell-line-mean floor; same features and `y_clip` as elsewhere.
Drugs whose SMILES give no scaffold are assigned to their own group.

In [2]:
import numpy as np, pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr

ROOT = Path(r"D:\Pancreatic_GraphTCDR"); P = ROOT/"data"/"processed"; T = P/"target_v2"; RES = ROOT/"results"/"v2"
train = pd.read_csv(T/"train_v2.csv"); val = pd.read_csv(T/"val_v2.csv"); test = pd.read_csv(T/"test_v2.csv")
allp = pd.concat([train, val, test], ignore_index=True)
Z402 = np.load(P/"ae_embeddings_402x512.npy").astype(np.float32)
fp = np.load(P/"drug_morgan_fingerprints_1228x2048.npy").astype(np.float32)
smi = pd.read_csv(P/"drug_smiles_1228.csv")

def scaffold(s, i):
    try:
        m = Chem.MolFromSmiles(s)
        sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m) if m else ""
    except Exception:
        sc = ""
    return sc if sc else f"NOSCAF_{i}"          # acyclic / failed -> own group

smi["scaffold"] = [scaffold(s, i) for i, s in enumerate(smi["canonical_smiles"])]
drug_scaf = smi["scaffold"].values
print("drugs:", len(smi), "| unique scaffolds:", len(set(drug_scaf)),
      "| drugs with own-group (no scaffold):", int(smi.scaffold.str.startswith("NOSCAF").sum()))
allp["scaffold"] = drug_scaf[allp.drug_index.values]

drugs: 1228 | unique scaffolds: 983 | drugs with own-group (no scaffold): 42


In [3]:
from sklearn.model_selection import GroupShuffleSplit
def X_of(df): return np.hstack([Z402[df.omics_index.values], fp[df.drug_index.values]])
N_TRAIN = 12929
rows = []
for seed in [0, 1, 2]:
    for name, groups in [("random-drug", allp.drug_index.values), ("scaffold", allp.scaffold.values)]:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.21, random_state=seed)
        a, b = next(gss.split(allp, groups=groups))
        tr = allp.iloc[a].sample(min(N_TRAIN, len(a)), random_state=seed); te = allp.iloc[b]
        assert set(tr.drug_index).isdisjoint(set(te.drug_index))
        if name == "scaffold": assert set(tr.scaffold).isdisjoint(set(te.scaffold))
        cm = tr.groupby("omics_index").y_clip.mean(); gm = tr.y_clip.mean()
        floor = te.omics_index.map(cm).fillna(gm).values
        rf = RandomForestRegressor(300, max_features="sqrt", min_samples_leaf=3, random_state=seed, n_jobs=-1)
        rf.fit(X_of(tr), tr.y_clip.values); p = rf.predict(X_of(te)); y = te.y_clip.values
        rows.append(dict(split=name, seed=seed, n_test=len(te), test_drugs=te.drug_index.nunique(),
                         RF_R2=r2_score(y, p), RF_MAE=mean_absolute_error(y, p), RF_rho=spearmanr(y, p)[0],
                         floor_R2=r2_score(y, floor)))
        print(rows[-1])
res = pd.DataFrame(rows)
print(res.groupby("split")[["RF_R2", "RF_MAE", "RF_rho", "floor_R2"]].agg(["mean", "std"]).round(3))
res.to_csv(RES/"scaffold_split_comparison.csv", index=False)

{'split': 'random-drug', 'seed': 0, 'n_test': 3935, 'test_drugs': 258, 'RF_R2': 0.07600218568223593, 'RF_MAE': 0.5574480157752942, 'RF_rho': np.float64(0.35013387639481913), 'floor_R2': 0.004781443379669681}
{'split': 'scaffold', 'seed': 0, 'n_test': 3293, 'test_drugs': 224, 'RF_R2': 0.05540372911633318, 'RF_MAE': 0.5248747811411418, 'RF_rho': np.float64(0.3547121169210468), 'floor_R2': 0.014679062888851235}
{'split': 'random-drug', 'seed': 1, 'n_test': 3870, 'test_drugs': 258, 'RF_R2': 0.2186011397487756, 'RF_MAE': 0.5117328415690964, 'RF_rho': np.float64(0.37632622241464553), 'floor_R2': 0.002046369630376743}
{'split': 'scaffold', 'seed': 1, 'n_test': 3693, 'test_drugs': 232, 'RF_R2': 0.06893177501749725, 'RF_MAE': 0.5240102011228297, 'RF_rho': np.float64(0.3033317028381573), 'floor_R2': 0.011169847326679805}
{'split': 'random-drug', 'seed': 2, 'n_test': 3702, 'test_drugs': 258, 'RF_R2': 0.16367556781597736, 'RF_MAE': 0.5188649936372773, 'RF_rho': np.float64(0.4148909492201066), 'flo

## Results (RF + AE, y_clip, 3 split seeds)

| Split | R² | MAE | Spearman |
|---|---|---|---|
| Random drug-grouped | 0.153 ± 0.072 | 0.529 ± 0.025 | 0.380 ± 0.033 |
| Bemis-Murcko scaffold | 0.066 ± 0.009 | 0.529 ± 0.009 | 0.332 ± 0.026 |

Scaffold R² is below random-drug R² in all three seeds; floor R² is about 0.01 in both. Signal survives scaffold
separation but is small. 983 scaffolds for 1,228 drugs; 42 acyclic drugs form their own groups.

Limitations: RF only; 3 seeds; test sets differ in size and composition; only the cold-drug ranking of RF was tested.

### 14.10 Data Integrity and Split Verification

#### Objective

Before interpreting the scaffold-based drug-response prediction results, it is important to verify the integrity of the source dataset and confirm that the original data partitions are consistent.

This audit examines the following aspects:

1. **Partition sizes:** Counts the observations, unique drugs and unique cell lines in the training, validation and test sets.
2. **Missing target values:** Checks whether any observations have missing `y_clip` values that could affect model training or evaluation.
3. **Drug–cell-line pair overlap:** Checks whether an identical drug–cell-line pair appears in more than one source partition.
4. **Pooled dataset consistency:** Verifies the total number of observations, unique drugs and assigned scaffold groups after combining the three partitions.
5. **Scaffold split verification:** Refers to the assertions in the experimental loop that check drug-level separation and scaffold-level separation for each random seed.

#### Rationale

These checks help identify basic data-integrity problems before model evaluation. In particular, exact drug–cell-line pair overlap across the original partitions could compromise evaluation if duplicate observations are treated as independent samples.

The checks also help confirm that the pooled dataset used for the scaffold experiment contains the expected number of observations and drug identifiers.

However, the absence of exact pair overlap does not establish the absence of every form of data leakage. It does not verify preprocessing independence, exclude chemical similarity across different scaffolds, or establish the biological reliability of the response labels.



In [4]:

# Audit 1: Verify the source partitions and pooled observations

parts = {
    "train": train,
    "val": val,
    "test": test,
}

for name, df in parts.items():
    print(
        name,
        "| rows:", len(df),
        "| drugs:", df["drug_index"].nunique(),
        "| cell lines:", df["omics_index"].nunique(),
        "| missing y_clip:", df["y_clip"].isna().sum(),
    )

# Check whether drug-cell-line pairs overlap between partitions.
# If repeated measurements are intentional, investigate them
# before treating every duplicate as data leakage.

for a_name, b_name in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test"),
]:
    a = parts[a_name]
    b = parts[b_name]

    a_pairs = set(zip(a["drug_index"], a["omics_index"]))
    b_pairs = set(zip(b["drug_index"], b["omics_index"]))

    overlap = a_pairs & b_pairs
    print(f"{a_name} vs {b_name}: {len(overlap)} shared pairs")

# Audit 2: Check the scaffold assignments and split assertions

print("Pooled rows:", len(allp))
print("Unique drugs:", allp["drug_index"].nunique())
print("Unique scaffolds:", allp["scaffold"].nunique())

for seed in [0, 1, 2]:
    print(f"Seed {seed}: split assertions passed in the existing loop")


train | rows: 12929 | drugs: 883 | cell lines: 28 | missing y_clip: 0
val | rows: 1306 | drugs: 99 | cell lines: 28 | missing y_clip: 0
test | rows: 3802 | drugs: 246 | cell lines: 28 | missing y_clip: 0
train vs val: 0 shared pairs
train vs test: 0 shared pairs
val vs test: 0 shared pairs
Pooled rows: 18037
Unique drugs: 1228
Unique scaffolds: 983
Seed 0: split assertions passed in the existing loop
Seed 1: split assertions passed in the existing loop
Seed 2: split assertions passed in the existing loop


Partition-level checks

The training set contains 12,929 observations involving 883 drugs and 28 cell lines. The validation set contains 1,306 observations involving 99 drugs and the same 28 cell lines. The test set contains 3,802 observations involving 246 drugs and 28 cell lines.

No missing y_clip values were detected in any of the three partitions. This indicates that the target column is complete for the observations examined in this audit.

Drug–cell-line pair overlap

No exact drug–cell-line pair overlap was detected between the training, validation and test partitions. This is a positive data-integrity result because it indicates that the same indexed drug–cell-line combination was not duplicated across these partitions.

However, different drugs may still share chemical characteristics, and cell lines are represented across the partitions. These observations are not, by themselves, evidence of leakage in a drug-response prediction task.

Pooled dataset consistency

After combining the three partitions, the dataset contains 18,037 observations, 1,228 unique drugs and 983 assigned scaffold groups. These counts are consistent with the dataset used in the scaffold comparison experiment.

Scaffold split checks

The original experimental loop contains assertions to verify that training and test drugs are disjoint and, for scaffold evaluation, that their assigned scaffold groups are also disjoint. Successful completion of these assertions provides evidence that the corresponding split constraints were satisfied.

The printed messages for seeds 0, 1 and 2 only report the seeds used; they do not independently rerun or verify the assertions.

Overall conclusion

The audit did not identify missing target values or exact drug–cell-line pair overlap across the original data partitions. The pooled dataset dimensions are also consistent with the expected dataset.

These results support the basic integrity of the source data. Further checks are still required to establish whether feature preprocessing was independent of the held-out observations and whether the evaluation supports the intended generalization claims.

### 14.11 Autoencoder Embedding Provenance Audit

#### Objective

The Random Forest model uses autoencoder-derived omics embeddings stored in `ae_embeddings_402x512.npy`. Before interpreting the model's generalization performance, it is necessary to understand how these embeddings were generated.

This audit aims to identify the source data used to train the autoencoder, the preprocessing steps applied to the omics features, and whether information from held-out observations was used during feature construction.

#### Rationale

An autoencoder learns a compressed representation of input features. In this project, the resulting embeddings are combined with drug molecular fingerprints to predict drug-response labels.

The evaluation depends partly on how the autoencoder was fitted:

- **Training-only feature fitting:** The autoencoder is fitted using only the permitted training cell-line features.
- **Transductive feature fitting:** The autoencoder uses features from cell lines that may also occur in the evaluation set, without using their response labels.
- **Potential information leakage:** Held-out response labels or other evaluation-dependent information influence feature construction, preprocessing decisions or model selection.

These settings represent different assumptions about what information is available when predictions are made. Using unlabeled features from evaluation cell lines is not automatically leakage, but it must be documented because it changes the generalization setting.

#### Audit procedure

The code and notebook cells responsible for creating `ae_embeddings_402x512.npy` will be examined to determine:

1. Which omics dataset was used as input.
2. Whether normalization or other preprocessing was performed.
3. Which observations were used to fit the autoencoder.
4. How the 512-dimensional embeddings were generated and saved.
5. Whether any response labels or test-set outcomes influenced the process.

The audit will establish the preprocessing assumptions before deciding whether any changes to the existing experiment are necessary.

### 14.11 Loading Processed Features and Generating Drug Scaffolds

#### Objective

This section loads the processed drug-response dataset, cell-line embeddings, molecular fingerprints and drug SMILES representations required for the scaffold-based evaluation.

The original training, validation and test partitions are combined to construct a common dataset for the subsequent split comparison. Bemis–Murcko scaffolds are then generated from the canonical SMILES representations to group drugs according to their assigned structural frameworks.

#### Methodology

The following operations are performed:

1. **Dataset loading:** The training, validation and test CSV files are loaded and combined into a single DataFrame containing the available drug–cell-line response observations.
2. **Omics feature loading:** The saved autoencoder embeddings are loaded from `ae_embeddings_402x512.npy` and converted to 32-bit floating-point values.
3. **Molecular feature loading:** Precomputed Morgan fingerprints are loaded from `drug_morgan_fingerprints_1228x2048.npy`.
4. **Chemical structure loading:** Canonical SMILES representations are read from `drug_smiles_1228.csv`.
5. **Scaffold generation:** RDKit is used to generate a Bemis–Murcko scaffold for each drug whose molecular structure can be parsed.
6. **Handling unavailable scaffolds:** Drugs for which a non-empty scaffold is not obtained are assigned individual identifiers using the `NOSCAF` prefix.
7. **Mapping scaffolds to observations:** Each pooled drug–cell-line observation is assigned the scaffold corresponding to its `drug_index`.

#### Rationale

Grouping observations by scaffold enables an evaluation in which drugs sharing the same assigned scaffold can be held out together. This provides a stricter test of generalization than a random drug-grouped split when related compounds share a scaffold.

The embeddings and fingerprints are loaded as previously generated features. This section does not establish how the autoencoder was trained or whether its preprocessing was independent of the evaluation data. Those details must be examined separately.

The pooled dataset is used to construct the scaffold comparison; the resulting split must enforce the required drug and scaffold separation explicitly.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr

ROOT = Path(r"D:\Pancreatic_GraphTCDR")
P = ROOT / "data" / "processed"
T = P / "target_v2"
RES = ROOT / "results" / "v2"

train = pd.read_csv(T / "train_v2.csv")
val = pd.read_csv(T / "val_v2.csv")
test = pd.read_csv(T / "test_v2.csv")

allp = pd.concat([train, val, test], ignore_index=True)

Z402 = np.load(P / "ae_embeddings_402x512.npy").astype(np.float32)
fp = np.load(P / "drug_morgan_fingerprints_1228x2048.npy").astype(np.float32)
smi = pd.read_csv(P / "drug_smiles_1228.csv")

def scaffold(s, i):
    try:
        m = Chem.MolFromSmiles(s)
        sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m) if m else ""
    except Exception:
        sc = ""
    return sc if sc else f"NOSCAF_{i}"

smi["scaffold"] = [
    scaffold(s, i) for i, s in enumerate(smi["canonical_smiles"])
]

drug_scaf = smi["scaffold"].values

print(
    "Drugs:", len(smi),
    "| Unique scaffolds:", len(set(drug_scaf)),
    "| Drugs without scaffold:", int(smi["scaffold"].str.startswith("NOSCAF").sum())
)

allp["scaffold"] = drug_scaf[allp["drug_index"].values]

Drugs: 1228 | Unique scaffolds: 983 | Drugs without scaffold: 42


The dataset contains 1,228 drugs grouped into 983 scaffold groups. Forty-two drugs have no valid scaffold and are assigned individual groups to keep them separate during splitting.

Scaffold-based train/validation/test split

In [7]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 0
groups = allp["scaffold"].values

split1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(
    split1.split(allp, groups=groups)
)

temp = allp.iloc[temp_idx]
temp_groups = temp["scaffold"].values

split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(
    split2.split(temp, groups=temp_groups)
)

train_scaf = allp.iloc[train_idx].copy()
val_scaf = temp.iloc[val_rel].copy()
test_scaf = temp.iloc[test_rel].copy()

assert set(train_scaf["scaffold"]).isdisjoint(val_scaf["scaffold"])
assert set(train_scaf["scaffold"]).isdisjoint(test_scaf["scaffold"])
assert set(val_scaf["scaffold"]).isdisjoint(test_scaf["scaffold"])

print("Train:", train_scaf.shape)
print("Validation:", val_scaf.shape)
print("Test:", test_scaf.shape)
print("Scaffold overlap: none")

Train: (14873, 14)
Validation: (1609, 14)
Test: (1555, 14)
Scaffold overlap: none





The split keeps each scaffold group in only one partition, reducing chemical-structure overlap between training and testing. The printed sizes show the resulting partition sizes.